# 13 — Shift-aware set-valued decoding (không train, chỉ CPU)

**Vì sao.** MILK10k chấm **từng lớp như một bài toán nhị phân riêng** ở ngưỡng 0.5, nên một lesion được phép dương ở nhiều
lớp. M0 đang giải mã top-1 (đúng 1 lớp/lesion). Leaderboard cho thấy AUC của M0 (0.925) ngang/hơn các đội đạt 0.61 nhưng
Dice thấp hơn ~0.09: điểm mất ở **bước quyết định**, không phải ở mô hình (vd test BCC bị dự đoán ~2 lần số ca thật do
label shift; DF có AUC 0.99 nhưng Dice 0.56).

**Bộ giải mã** (chỉ dùng OOF có nhãn + xác suất trên ảnh test *không nhãn*, không train lại):
```
p (TB 5 fold) ─ calib vector scaling (fit OOF) ─ BBSE ước lượng prior test π̂ ─ đi nửa bước π½ ∝ √(π̂·π_train)
  ─ q = calib(p)·π½/π_train ─ mỗi lớp: chọn top-k theo q_c, k cực đại F1 kỳ vọng ─ ∪ argmax(p) ─ file nộp
```
So sánh 4 cách: `raw` (top-1 như cũ) · `ef1_cal` · `ef1_bbse_half` · `ef1_bbse`.

**Chọn bằng mô phỏng cross-fit trên OOF, chốt trước khi nộp:** giữ 1 fold, fit mọi thứ trên 4 fold còn lại, rút mẫu giả-test
479 lesion theo các prior: không lệch · π̂ test (BBSE, không nhãn) · 4 prior ngẫu nhiên. **Luật:** chỉ chọn một cách khác
`raw` nếu Δ TB khi lệch ≥ **+0.01** *và* Δ khi không lệch ≥ **−0.005**; không đạt thì giữ `raw`.
**Mỗi nhóm đúng 1 file nộp** (`submissions/<nhóm>__cv5__<cách>/`); thư mục `_khac/` để tham khảo, **không nộp**.

Chạy được trên runtime CPU (không cần GPU), ~5–10 phút cho M0.

In [ ]:
# ===================== CẤU HÌNH =====================
REPO_URL = "https://github.com/trong5nhan6/fussion.git"
BRANCH = "main"
# Thư mục chứa run 5 fold (oof.csv + test_probs.csv + metrics.json). Thêm stage 9 / 10 nếu muốn so sánh nhiều nhóm.
RUN_DIRS = ["/content/drive/MyDrive/milk10k_ablation/outputs"]
GROUPS = ["swin_base_384__abl_M0"]      # nhóm cần giải mã (đủ 5 fold); [] = mọi nhóm đủ 5 fold trong RUN_DIRS
OUT_DIR = "/content/drive/MyDrive/milk10k_shift_decode"
REPS = 100                              # số mẫu giả-test mỗi fold mỗi kịch bản (100 -> ~5 phút / nhóm)
N_RANDOM = 4                            # số prior ngẫu nhiên trong mô phỏng

## 1. Drive + code

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q "timm>=1.0" openpyxl
!git log -1 --format="Commit: %h %s"
for d in RUN_DIRS:
    runs = [x for x in os.listdir(d) if any(x.startswith(g) for g in GROUPS)] if GROUPS else os.listdir(d)
    print(d, "->", len(runs), "run:", sorted(runs)[:10])

## 2. Mô phỏng + giải mã

In [ ]:
groups_arg = ("--groups " + " ".join(GROUPS)) if GROUPS else ""
dirs_arg = " ".join(f'"{d}"' for d in RUN_DIRS)
!python scripts/shift_decode.py --dirs {dirs_arg} {groups_arg} --out outputs/shift_decode --reps {REPS} --random {N_RANDOM}

## Kết quả
Bảng Δ: so với `raw` (top-1 như file nộp 0.519), ghép cặp trên cùng mẫu giả-test. Cột `đạt luật` cho biết cách nào được
phép chọn. Bảng "số lesion dương" trên test thật giúp kiểm tra cảm quan: số ca BCC gần như giữ nguyên (nhãn argmax luôn được giữ), còn các lớp đang bị dự đoán
thiếu (AKIEC, BKL, INF, MAL_OTH...) được thêm nhãn dương; mỗi lesion có TB ~1.3 nhãn dương.

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("outputs/shift_decode/shift_decode.md", encoding="utf-8").read()))
!mkdir -p "{OUT_DIR}" && rsync -a outputs/shift_decode/ "{OUT_DIR}/"
import glob
print("FILE NỘP (1 file / nhóm):")
for f in sorted(glob.glob(f"{OUT_DIR}/submissions/*/submission.csv")):
    if "/_khac/" not in f:
        print("  ", f)

In [ ]:
# Kiểm tra định dạng file nộp: 479 lesion, 11 lớp, mỗi lesion >= 1 lớp > 0.5, điểm trong [0, 1]
import pandas as pd
for f in sorted(glob.glob(f"{OUT_DIR}/submissions/*/submission.csv")):
    if "/_khac/" in f:
        continue
    s = pd.read_csv(f); pos = s.iloc[:, 1:] > 0.5
    print(f.split("/")[-2], "|", len(s), "lesion |", s.shape[1] - 1, "lớp | lesion không có lớp dương:",
          int((pos.sum(1) == 0).sum()), "| nhãn dương TB/lesion:", round(pos.sum(1).mean(), 2),
          "| min/max:", round(s.iloc[:, 1:].values.min(), 3), round(s.iloc[:, 1:].values.max(), 3))